Aquí un ejemplo básico de Policy Gradients con REINFORCE, usando PyTorch y OpenAI Gym en el entorno "CartPole-v1":

In [1]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
# 1. Crear el entorno de OpenAI Gym
env = gym.make("CartPole-v1")

/usr/local/lib/python3.11/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.11/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [3]:
# 2. Definir la red neuronal para la política
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(PolicyNetwork, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim),
            nn.Softmax(dim=-1)  # Distribución de probabilidad sobre acciones
        )

    def forward(self, x):
        return self.fc(x)

In [4]:
# 3. Inicializar la política y optimizador
policy = PolicyNetwork(state_dim=4, action_dim=2)
optimizer = optim.Adam(policy.parameters(), lr=0.01)

In [5]:
# 4. Función para obtener acciones
def get_action(state):
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0) # Add unsqueeze(0)
    probs = policy(state)
    action = np.random.choice(len(probs.detach().numpy()), p=probs.detach().numpy())
    return action

In [12]:
# 5. Entrenamiento con Policy Gradient (REINFORCE)
def train_policy_gradient(episodes=1000, gamma=0.99):
    for episode in range(episodes):
        state = env.reset()[0]
        rewards = []
        log_probs = []

        # Simulación de un episodio
        while True:
            # Convertir el estado a tensor con las dimensiones correctas
            current_state_tensor = torch.tensor(state, dtype=torch.float32)

            # Asegurar que el estado tenga la forma correcta para la red
            if len(current_state_tensor.shape) == 1:
                current_state_tensor = current_state_tensor.unsqueeze(0)  # Añadir dimensión de batch

            # Obtener acción y probabilidades
            action_probs = policy(current_state_tensor)
            action = torch.multinomial(action_probs, 1).item()

            # Ejecutar acción en el entorno
            next_state, reward, done, _, _ = env.step(action)

            # Obtener log probabilidad de la acción tomada
            log_prob = torch.log(action_probs.squeeze(0)[action])

            rewards.append(reward)
            log_probs.append(log_prob)
            state = next_state

            if done:
                break

        # Calcular recompensas descontadas
        G = []
        total_return = 0
        for r in reversed(rewards):
            total_return = r + gamma * total_return
            G.insert(0, total_return)

        # Convertir a tensor y normalizar
        G = torch.tensor(G, dtype=torch.float32)
        if len(G) > 1:
            G = (G - G.mean()) / (G.std() + 1e-9)  # Evitar división por cero

        # Calcular pérdida
        loss = -torch.sum(torch.stack(log_probs) * G)

        # Paso de optimización
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if episode % 100 == 0:
            print(f"Episode {episode}, Total Reward: {sum(rewards)}")

In [13]:
# 6. Ejecutar el entrenamiento
train_policy_gradient()

RuntimeError: both arguments to linear need to be at least 1D, but they are 0D and 2D